# Video RAG — Phase 1: Notebook 05
## Quantitative Baseline Evaluation Benchmark

This notebook runs the formal evaluation benchmark on the annotated dataset:
- **Retrieval Metrics**: Recall@1, Recall@3, Recall@5, MRR
- **Temporal Grounding Metrics**: Temporal IoU, Top-1 Overlap Hit Rate
- **Answer Quality Metrics**: Abstention Accuracy for unanswerable questions
- **System Metrics**: Average Retrieval Latency, Generation Latency, Total Latency

Results are exported to `experiments/results/baseline_results.json` and `.csv`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))

from src.rag_pipeline import VideoRAGPipeline
from src.evaluation import RAGEvaluator


### Step 1: Initialize Pipeline & Load Dataset


In [ ]:
pipeline = VideoRAGPipeline()
pipeline.load_index("../data/vector_store")
pipeline.current_video = "sample.mp4"

dataset_path = "../data/evaluation/benchmark_questions.json"
evaluator = RAGEvaluator(output_dir="../experiments/results")
questions = evaluator.load_evaluation_dataset(dataset_path)

print(f"Loaded {len(questions)} evaluation questions from {dataset_path}")


### Step 2: Run Full Benchmark


In [ ]:
# Execute evaluation across all benchmark questions with top_k=5
results_summary = pipeline.run_evaluation(
    eval_dataset_path=dataset_path,
    experiment_name="baseline_results",
    top_k=5
)


### Step 3: Evaluation Summary Metrics


In [ ]:
print("=" * 60)
print("        VIDEO RAG PHASE 1 — EVALUATION RESULTS")
print("=" * 60)
print(f"Total Questions Evaluated:    {results_summary['counts']['total_queries']}")
print(f"  - Answerable Queries:       {results_summary['counts']['answerable_queries']}")
print(f"  - Unanswerable Queries:     {results_summary['counts']['unanswerable_queries']}")
print("-" * 60)
print("RETRIEVAL METRICS:")
for k, v in results_summary['retrieval_metrics'].items():
    print(f"  {k:20s}: {v}")

print("-" * 60)
print("TEMPORAL GROUNDING METRICS:")
for k, v in results_summary['temporal_grounding_metrics'].items():
    print(f"  {k:20s}: {v}")

print("-" * 60)
print("ANSWER & ABSTENTION METRICS:")
for k, v in results_summary['answer_metrics'].items():
    print(f"  {k:20s}: {v}")

print("-" * 60)
print("LATENCY (Seconds):")
for k, v in results_summary['latency_metrics_sec'].items():
    print(f"  {k:20s}: {v}s")
print("=" * 60)


### Step 4: Detailed Sample Results Table


In [ ]:
df_results = pd.read_csv("../experiments/results/baseline_results.csv")
display_cols = [
    "sample_id", "question", "answerable", "retrieval_hit_at_1", 
    "top1_temporal_iou", "abstention_correct", "total_latency_sec"
]
df_results[display_cols]


### Step 5: Metric Visualizations


In [ ]:
# Plot Retrieval & Temporal Performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Retrieval Recall & MRR
ret_metrics = results_summary['retrieval_metrics']
axes[0].bar(list(ret_metrics.keys()), list(ret_metrics.values()), color="#2b5c8f", edgecolor="black")
axes[0].set_ylim(0, 1.1)
axes[0].set_title("Retrieval Performance (Recall@K & MRR)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Score")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)
for i, (k, v) in enumerate(ret_metrics.items()):
    if v is not None:
        axes[0].text(i, v + 0.03, f"{v:.2f}", ha="center", fontweight="bold")

# Plot 2: Latency Breakdown
lat_metrics = results_summary['latency_metrics_sec']
axes[1].bar(["Retrieval", "Generation", "Total"], list(lat_metrics.values()), color="#3f9b0b", edgecolor="black")
axes[1].set_title("Average Latency Breakdown (Seconds)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Seconds")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)
for i, (k, v) in enumerate(lat_metrics.items()):
    axes[1].text(i, v + 0.01, f"{v:.3f}s", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../experiments/results/evaluation_plot.png", dpi=150)
plt.show()
